# Diabetes Hospital Readmission - Preprocessing

The preprocessing step for the diabetes walkthrough, mirroring `U1_RealEstate-2_Preprocess`:
drop leakage columns, encode every categorical, and save one modeling-ready CSV that notebooks
3-7 load. As with the real-estate data we do **no scaling here** - scaling happens inside the
notebooks that need it, fitted on training data only.

## Learning objectives

- Identify and drop columns that leak the target
- Ordinal-encode a categorical whose levels have a genuine order
- Dummy-encode unordered categoricals, dropping one level per feature to limit collinearity
- Explain why scaling belongs in the modeling notebooks rather than here
- Write a single modeling-ready CSV consumed by the rest of the spine

## Background

This notebook assumes the dataset and its 11% positive rate from `U1_Diabetes-1_EDA`, and the
encoders from `U1-2_Preprocess` — `LabelEncoder` versus `OneHotEncoder`, and when each is
appropriate. Here those choices are made column by column on real data.

One convention worth stating up front: **no scaling happens in this notebook.** A scaler must be
fit on the training split only, and the split does not exist yet, so scaling is deferred to each
modeling notebook. Saving a scaled CSV would leak test statistics into every downstream notebook at
once.

**Prerequisites:** `U1_Diabetes-1_EDA` (the dataset and its target),
`U1-2_Preprocess` (ordinal versus one-hot encoding), `U1-4_FeatSelect-1_VIF` (why dropping a
dummy level limits collinearity)

**Dataset:** `fairlearn.datasets.fetch_diabetes_hospital`, written out to
`Datasets/Diabetes/diabetes_hospital_preprocessed.csv`.

**References:** https://scikit-learn.org/stable/modules/preprocessing.html

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from tqdm import tqdm

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Load the data

The data is fetched from `fairlearn` rather than read from disk, so this notebook is
self-contained: there is no raw CSV to keep in sync, and anyone running it gets the same source
data. The cost is that it needs network access on first call.

Note that we start from the **raw** fetch rather than from anything `U1_Diabetes-1_EDA` produced.
The EDA notebook only looked; it changed nothing. All the actual transformation happens here, which
keeps a single, auditable path from source to modeling table.


In [2]:
from fairlearn.datasets import fetch_diabetes_hospital

data = fetch_diabetes_hospital(as_frame=True)
df = pd.concat([data.data, data.target], axis=1)
print(df.shape)

(101766, 25)


## 2. Drop leakage columns

`readmitted` (3-level) and `readmit_binary` are alternative encodings of the outcome - keeping
them as features would hand the model the answer. The target `readmit_30_days` stays.

Leakage is the failure where a feature encodes the answer. Here `readmitted` (three levels) and
`readmit_binary` are alternative encodings of the very outcome being predicted, so a model given
them would score near-perfectly and learn nothing — and would collapse in production, where those
columns do not exist at prediction time.

In [3]:
df = df.drop(columns=['readmitted', 'readmit_binary'])
df.head()

,race,gender,age,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,primary_diagnosis,number_diagnoses,max_glu_serum,A1Cresult,insulin,change,diabetesMed,medicare,medicaid,had_emergency,had_inpatient_days,had_outpatient_days,readmit_30_days
0,Caucasian,Female,'30 years or younger',Other,Referral,1,Other,41,0,1,Diabetes,1,None,None,No,No,No,False,False,False,False,False,0
1,Caucasian,Female,'30 years or younger','Discharged to Home',Emergency,3,Missing,59,0,18,Other,9,None,None,Up,Ch,Yes,False,False,False,False,False,0
2,AfricanAmerican,Female,'30 years or younger','Discharged to Home',Emergency,2,Missing,11,5,13,Other,6,None,None,No,No,Yes,False,False,False,True,True,0
3,Caucasian,Male,'30-60 years','Discharged to Home',Emergency,2,Missing,44,1,16,Other,7,None,None,Up,Ch,Yes,False,False,False,False,False,0
4,Caucasian,Male,'30-60 years','Discharged to Home',Emergency,1,Missing,51,0,8,Other,5,None,None,Steady,Ch,Yes,False,False,False,False,False,0


## 3. Ordinal-encode age

The age buckets have a natural order, so an integer encoding preserves information a dummy
encoding would throw away.

Age buckets are genuinely **ordered**: "30 years or younger" < "30-60 years" < "Over 60 years". An
integer encoding preserves that order and costs one column. Dummy-encoding would spend three
columns and discard the ordering — the exact case `U1-2_Preprocess` identified as belonging to
`LabelEncoder` rather than `OneHotEncoder`.

The mapping is built from an explicit `age_order` list rather than left to the encoder, for the same
reason as the real-estate quality ratings: an automatic encoding would order the levels
alphabetically, which for these labels is meaningless.

One property of ordinal encoding worth stating plainly. Mapping to 0, 1, 2 asserts not only an order
but an even **spacing** — it says the step from "30 or younger" to "30–60" is the same size as the
step from "30–60" to "over 60". That is almost certainly false in clinical terms. The encoding is
accepted anyway because the ordering it preserves is worth more than the false spacing costs, and
because tree models — which do most of the work in this spine — only ever compare values, never
their differences.


In [4]:
age_order = ["'30 years or younger'", "'30-60 years'", "'Over 60 years'"]

df['age'] = df['age'].astype(str).map({v: i for i, v in enumerate(age_order)})
df['age'].value_counts().sort_index()

age
0     2509
1    30716
2    68541
Name: count, dtype: int64

## 4. Dummy-encode the remaining categoricals

Same convention as the real-estate preprocess: one dummy column per level, dropping the
most frequent level of each feature to reduce collinearity.

For unordered categoricals there is no defensible integer ordering, so each level becomes its own
0/1 column.

Note the extra step: for each feature the **most frequent level's** dummy column is dropped. With
all $k$ dummies present the set is perfectly collinear — they sum to 1 in every row, so any one is
exactly determined by the others, which is the infinite-VIF case from `U1-4_FeatSelect-1_VIF`.
Dropping one level per feature removes that dependency; the dropped level becomes the baseline that
the remaining coefficients are measured against.

Dropping the **most frequent** level, rather than the first alphabetically, is a small deliberate
choice. The dropped level becomes the implicit baseline that every remaining coefficient is measured
against, and the most common level is the most useful baseline: "compared to a typical patient" is a
more interpretable reference point than "compared to whichever category sorted first".

The width cost is real. Each categorical with $k$ levels contributes $k-1$ columns, and the diabetes
data has many such fields, which is how a few dozen clinical columns become a matrix hundreds wide.
That width is exactly what `U1_Diabetes-5_FeatSelect` is for — and the reason a wide, sparse,
mostly-binary matrix is the regime where gradient-boosted trees tend to beat neural networks, as
`U1_Diabetes-4_Keras` finds empirically.


In [5]:
cat_cols = df.select_dtypes(exclude='number').columns.tolist()
print("categorical columns:", cat_cols)

X_dum = pd.DataFrame()
for c in cat_cols:
    dummy = pd.get_dummies(df[c].astype(str), prefix=c, dtype=int)
    col_to_drop = dummy.sum(axis=0).idxmax()
    dummy = dummy.drop(columns=[col_to_drop])
    X_dum = pd.concat([X_dum, dummy], axis=1)
# end

df = pd.concat([df.drop(columns=cat_cols), X_dum], axis=1)
print(df.shape)
df.head()

categorical columns: ['race', 'gender', 'discharge_disposition_id', 'admission_source_id', 'medical_specialty', 'primary_diagnosis', 'max_glu_serum', 'A1Cresult', 'insulin', 'change', 'diabetesMed', 'medicare', 'medicaid', 'had_emergency', 'had_inpatient_days', 'had_outpatient_days']


(101766, 42)


,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_diagnoses,readmit_30_days,race_AfricanAmerican,race_Asian,race_Hispanic,race_Other,race_Unknown,gender_Male,gender_Unknown/Invalid,discharge_disposition_id_Other,admission_source_id_Other,admission_source_id_Referral,medical_specialty_Cardiology,medical_specialty_Emergency/Trauma,medical_specialty_Family/GeneralPractice,medical_specialty_InternalMedicine,medical_specialty_Other,primary_diagnosis_'Genitourinary Issues',primary_diagnosis_'Musculoskeletal Issues',primary_diagnosis_'Respiratory Issues',primary_diagnosis_Diabetes,max_glu_serum_>200,max_glu_serum_>300,max_glu_serum_Norm,A1Cresult_>7,A1Cresult_>8,A1Cresult_Norm,insulin_Down,insulin_Steady,insulin_Up,change_Ch,diabetesMed_No,medicare_True,medicaid_True,had_emergency_True,had_inpatient_days_True,had_outpatient_days_True
0,0,1,41,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,0,3,59,0,18,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0
2,0,2,11,5,13,6,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
3,1,2,44,1,16,7,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0
4,1,1,51,0,8,5,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0


## 5. Save the preprocessed dataset

Notebooks 3-7 of this walkthrough load this CSV exactly the way the real-estate notebooks load
theirs.

One CSV, five consumers: notebooks 3 through 7 all read this file rather than repeating the encoding
work. That is worth doing for consistency as much as for speed — if each notebook encoded the
categoricals itself, a small difference in one of them would make their results quietly
incomparable.

The trade-off is a dependency. **Re-running this notebook changes the input to every downstream
notebook**, so their stored outputs become stale until they are re-run too. Any time the encoding
here changes, the whole spine needs re-executing in order.


In [6]:
import os

data_folder = 'C:/Users/Graham West/Python Notebooks/Meharry Teaching/Datasets/'
os.makedirs(data_folder + 'Diabetes', exist_ok=True)

out_path = data_folder + 'Diabetes/diabetes_hospital_preprocessed.csv'
df.to_csv(out_path, index=False)

print(f"saved: {out_path}")
print(f"shape: {df.shape}")
print(f"positive rate: {df['readmit_30_days'].mean():.1%}")

saved: C:/Users/Graham West/Python Notebooks/Meharry Teaching/Datasets/Diabetes/diabetes_hospital_preprocessed.csv
shape: (101766, 42)
positive rate: 11.2%


## 6. Review

- **Drop anything that encodes the target.** `readmitted` and `readmit_binary` are the outcome in
  other clothes; keeping them would produce a model that scores well and predicts nothing.
- **Ordinal-encode ordered categoricals** — age here — to preserve the ordering in one column.
- **Dummy-encode unordered ones**, dropping one level per feature so the dummy set is not perfectly
  collinear. The dropped level is the baseline.
- **Scaling is deliberately not done here.** A scaler must be fit on the training split, which does
  not exist yet; scaling before saving would leak test statistics into every downstream notebook.
- The saved CSV is the single input for notebooks 3 through 7, so re-running this notebook changes
  all of them.